# Modelling network belief propagation 

## Initialization

Loading the `vis.js` library for data visualization and the network simulation implementation from `network-based.fsx` file.

In [ ]:
#!js
// Load the 'vis' library so that we can access it in generated JS code
req = interactive.configureRequire({
    paths: { vis: "https://visjs.github.io/vis-network/standalone/umd/vis-network.min.js" } });
vis = null; req(["vis"], v => { vis = v; })

In [ ]:
#load "network-based.fsx"
open NetworkBased

## Initialization and visualization

In [ ]:
// Generate network with 8 agents and 10 links
let g = Graph.initGraph 8 20

In [ ]:
// Visualize the network using vis.js
Vis.visualizeNetwork g |> HTML

In [ ]:
// For each domain of beliefs, display the network with just beliefs from the domain
[ for domain in ideas -> Vis.visualizeNetwork (Vis.filterGraph domain g) ]
|> String.concat "" |> HTML

## Primitive operations

### Adopting a belief
*"You are friends with Liverpool fans and they convince you to also vote for Greens"*

Find an agent such that it has more than two neighbours that believe in a shared belief (i.e., have links labelled with this belief) from a domain that the given agent does not have any beliefs about and adopt this belief (add edges with the two or more neighbours that beleive it)

In [ ]:
// Single-step operation of the simulation
let update = 
  // Generate all potential (agent, new belief, neightbours) combinations
  Logic.withOne Sim.getAgentsWithBeliefsToAdopt
    // Pick one using 'withOne' and addopt it (add the links)
    Sim.adoptBelief

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Adding 21<--(A3)-->23 Adding 21<--(A3)-->23 Adding 25<--(C1)-->27 Adding 25<--(C1)-->24

### Creating new connection
*"You randomly run into another fan of The Fugs and you become friends."*

Find a pair of agents that are distinct & disconnected and have a shared belief. Add a new link connecting the two agents.

In [ ]:
let update = 
  Logic.withOne Sim.getDisconnectedCompatibleAgents
    Sim.addEdge

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Adding 22<--(A3)-->24 Adding 26<--(A2)-->23 Adding 21<--(C1)-->24 Adding 26<--(B1)-->28 Adding 22<--(A3)-->25

### Change a belief

*"You are connected to someone because you both liked LibDems, but all your friends became Tory voters so you change your mind."*

Given any existing edge, look at the beliefs from the same domain of the two agents and change the belief on the edge to any of the possible beliefs. Note that this is done by picking an edge, but picking an agent and then an edge would be the same.

In [ ]:
let update = 
  Logic.withOne Sim.getEdges 
    Sim.updateBeliefOnEdge

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Adopting B1 on 21<--(B1)-->26 Adopting A2 on 23<--(A4)-->24 Adopting C1 on 21<--(C1)-->26 Adopting B2 on 26<--(B2)-->27 Adopting A2 on 24<--(A1)-->27

### Remove a belief

*"You are connected to someone because of your political opinion, but you keep arguing so you stop caring about politics."*

Find a conflicting edge, i.e., an edge where both agents also believe in other beliefs (are connected via other beliefs) and remove the edge altogether.

In [ ]:
let update = 
  Logic.withOne Sim.getConflictingEdges 
    Sim.removeEdge

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Removing 24<--(C1)-->27 Removing 21<--(B2)-->23 Removing 26<--(B2)-->27 Removing 23<--(A2)-->24 Removing 23<--(A3)-->25

## Running the simulation

In [ ]:
let update = 
  // Use 'applyOne' to specify that, in each turn, one of the operations
  // should be randomly picked and applied
  Logic.applyOne [
    Logic.withOne Sim.getAgentsWithBeliefsToAdopt
      Sim.adoptBelief
    Logic.withOne Sim.getDisconnectedCompatibleAgents
      Sim.addEdge
    Logic.withOne Sim.getEdges 
      Sim.updateBeliefOnEdge
    Logic.withOne Sim.getConflictingEdges 
      Sim.removeEdge
  ]

// Run 100 iterations of the simulation
let t, log = Sim.iterateWithLog 100 update g
// Draw every 20th iteration (by choosing networks at index i*20 from the trace 't')
String.concat "" [ for i in 0 .. 5 -> Vis.visualizeNetwork t.[i*20] ] + log |> HTML

Adding 25<--(C1)-->27 Adding 25<--(C1)-->24 Adding 27<--(B1)-->21 Adopting B3 on 25<--(B1)-->27 Adding 21<--(A2)-->26 Adding 21<--(A2)-->23 Adding 21<--(A2)-->26 Adding 21<--(A2)-->26 Adding 21<--(A2)-->23 Adopting B3 on 21<--(B1)-->26 Adopting B1 on 21<--(B2)-->23 Removing 25<--(B3)-->27 Adopting B1 on 21<--(B1)-->23 Removing 24<--(C1)-->27 Adopting C1 on 24<--(C1)-->25 Removing 26<--(B2)-->27 Adopting C1 on 21<--(C1)-->26 Removing 24<--(A1)-->27 Removing 26<--(B3)-->27 Adding 24<--(B1)-->28 Adding 28<--(A2)-->24 Adding 28<--(A2)-->21 Adding 21<--(C1)-->24 Adding 28<--(C3)-->24 Adding 28<--(C3)-->24 Removing 22<--(A3)-->23 Removing 21<--(A2)-->23 Adopting A2 on 24<--(A2)-->28 Adopting C3 on 21<--(C1)-->24 Adopting C3 on 21<--(C3)-->24 Removing 23<--(A2)-->24 Removing 24<--(C3)-->28 Adding 25<--(C1)-->21 Removing 21<--(B1)-->23 Adding 25<--(B1)-->28 Adopting B3 on 21<--(B1)-->28 Removing 24<--(A2)-->26 Adopting B1 on 24<--(B1)-->25 Removing 23<--(A4)-->24 Adopting C1 on 25<--(C1)-->27 Adopting C1 on 21<--(C3)-->24 Adopting C1 on 21<--(C1)-->26 Adding 24<--(C1)-->26 Adding 23<--(B1)-->24 Adding 23<--(B1)-->25 Adding 28<--(C3)-->24 Adding 28<--(C3)-->24 Adding 25<--(C1)-->26 Removing 23<--(C3)-->24 Adding 28<--(A2)-->26 Adding 27<--(C1)-->26 Adding 27<--(B1)-->23 Removing 21<--(B3)-->26 Adding 23<--(B1)-->21 Removing 24<--(A2)-->28 Adding 28<--(B1)-->23 Adopting B1 on 23<--(B1)-->24 Removing 21<--(C1)-->24 Adding 23<--(C1)-->21 Adding 23<--(C1)-->27 Adding 23<--(C1)-->24 Adding 23<--(C1)-->25 Adding 23<--(C1)-->25 Adding 26<--(C1)-->23 Adopting C1 on 21<--(C1)-->25 Adding 26<--(B3)-->28 Adding 26<--(B3)-->21 Adding 26<--(B3)-->21 Adding 26<--(B3)-->21 Adding 26<--(B3)-->21 Adding 26<--(B3)-->21 Adding 27<--(B1)-->28 Adding 21<--(C1)-->24 Removing 21<--(B3)-->28 Adopting C1 on 23<--(C1)-->24 Removing 24<--(B1)-->28 Removing 27<--(B1)-->28 Adopting C1 on 21<--(C1)-->26 Removing 23<--(C1)-->24 Adding 27<--(B1)-->28 Adopting B1 on 23<--(B1)-->28 Adopting C3 on 21<--(C1)-->24 Adopting C1 on 25<--(C1)-->27 Removing 21<--(C1)-->23 Adopting B3 on 25<--(B1)-->28 Removing 21<--(B3)-->26 Adopting B3 on 26<--(B3)-->28 Removing 21<--(C1)-->26 Adopting B3 on 25<--(B3)-->28 Removing 23<--(B1)-->28 Removing 24<--(C1)-->25 Adopting B3 on 26<--(B3)-->28 Adding 23<--(B1)-->28 Removing 23<--(B1)-->25 Adopting C1 on 23<--(C1)-->27

In [ ]:
// The above seems to be adding way too many edges - so to ballance that
// we can use 'applyOneProb' that lets us specify probability (set this 
// to higher for removing and edge) and we can also add a new rule using
// Sim.getEdges and Sim.removeEdge to just randomly remove an edge.
// If agent has no connection, none of the standard operations ever add
// it back to the network, so the below adds one more operation, which
// randomly connects disconnected agent to random agent in the network
let update = 
  Logic.applyOneProb [
    0.1, Logic.withOne Sim.getAgentsWithBeliefsToAdopt
      Sim.adoptBelief
    0.1, Logic.withOne Sim.getDisconnectedCompatibleAgents
      Sim.addEdge
    0.3, Logic.withOne Sim.getEdges 
      Sim.updateBeliefOnEdge
    0.3, Logic.withOne Sim.getConflictingEdges 
      Sim.removeEdge
    // Randomly remove edge to keep the average number of edges
    0.1, Logic.withOne Sim.getEdges 
      Sim.removeEdge
    // Connect agent that has been completely disconnected
    0.1, Logic.withOne Sim.getDisconnectedAgents (fun a1 ->
      Logic.withOne Sim.getConnectedAgents (fun a2 ->
        Logic.withOne (Sim.getAgentBeliefs a2.ID) (fun belief ->
          Sim.addEdge (a1.ID, a2.ID, belief))))
  ]

// Run 100 iterations of the simulation
let t, log = Sim.iterateWithLog 100 update g
// Draw every 20th iteration (by choosing networks at index i*20 from the trace 't')
String.concat "" [ for i in 0 .. 5 -> Vis.visualizeNetwork t.[i*20] ] + log |> HTML

Adopting A3 on 23<--(A4)-->24 Adding 21<--(A3)-->23 Adding 21<--(A3)-->23 Adding 21<--(B1)-->25 Adding 25<--(C3)-->23 Adding 25<--(C3)-->24 Removing 23<--(A3)-->24 Removing 26<--(B3)-->27 Adding 28<--(B1)-->26 Adding 21<--(C1)-->27 Adding 21<--(B1)-->24 Adding 28<--(C1)-->26 Adding 28<--(C1)-->21 Removing 21<--(B1)-->23 Removing 22<--(A3)-->23 Adding 22<--(B1)-->26 Adopting C3 on 24<--(C3)-->25 Adopting B1 on 21<--(B1)-->26 Removing 25<--(B1)-->27 Removing 26<--(C1)-->28 Removing 24<--(A3)-->27 Removing 22<--(B1)-->26 Adding 25<--(B1)-->28 Adopting B1 on 21<--(B1)-->24 Adding 23<--(B2)-->27 Adopting C1 on 21<--(C1)-->27 Removing 21<--(A3)-->23 Removing 21<--(B1)-->24 Adding 21<--(A3)-->25 Adding 21<--(A3)-->23 Removing 21<--(B2)-->23 Adding 24<--(B1)-->21 Adding 28<--(B1)-->24 Removing 21<--(B1)-->26 Removing 21<--(B1)-->25 Removing 23<--(B2)-->27 Adding 23<--(B1)-->21 Adding 23<--(B1)-->25 Adding 23<--(B1)-->24 Adding 23<--(B1)-->24 Adding 23<--(B1)-->25 Removing 21<--(A3)-->25 Adding 28<--(B1)-->23 Removing 23<--(A3)-->25 Removing 23<--(B1)-->25 Adding 22<--(C1)-->27 Adding 28<--(C1)-->27 Adopting B1 on 23<--(B1)-->24 Adopting A2 on 24<--(A1)-->27 Adopting B1 on 23<--(B1)-->28 Removing 24<--(C1)-->27 Adopting C3 on 24<--(C3)-->25 Adopting B2 on 26<--(B2)-->27 Adding 21<--(C1)-->22 Adding 28<--(A3)-->23 Adding 28<--(A3)-->21 Adding 28<--(A3)-->21 Adopting C1 on 22<--(C1)-->27 Adding 25<--(A3)-->28 Adding 25<--(A3)-->23 Removing 26<--(B1)-->28 Adopting B1 on 24<--(B1)-->28 Removing 23<--(A2)-->24 Adopting A3 on 23<--(A3)-->28 Adding 25<--(A3)-->21 Adding 22<--(C1)-->26 Adding 22<--(B2)-->26 Adding 22<--(B2)-->27 Adopting C1 on 21<--(C1)-->26 Removing 21<--(C1)-->26 Adopting B1 on 25<--(B1)-->28 Adding 26<--(C1)-->21 Adding 22<--(A2)-->26 Adding 22<--(A2)-->27 Adding 22<--(A2)-->26 Adding 22<--(A2)-->27 Removing 22<--(A2)-->26 Adopting C1 on 22<--(C1)-->26 Adopting C1 on 22<--(C1)-->27 Adding 28<--(C1)-->26 Adopting C1 on 21<--(C1)-->27 Removing 23<--(B1)-->28 Adopting A3 on 21<--(A3)-->25 Adopting C1 on 21<--(C1)-->28 Removing 23<--(B1)-->24 Adopting A3 on 21<--(A3)-->28 Adopting C3 on 23<--(C3)-->24 Adopting A2 on 24<--(A2)-->26 Adopting A3 on 21<--(A3)-->25 Adopting C3 on 23<--(C3)-->25

## Calculating network statistics

In [ ]:
#r "nuget: Plotly.NET"
#r "nuget: Plotly.NET.Interactive"
open Plotly.NET
open NetworkBased.Stats

Installed Packages Plotly.NET, 2.0.0 Plotly.NET.Interactive, 2.0.0

Loading extensions from `Plotly.NET.Interactive.dll`

## Average degree and clustering on single graph run sample
The two charts below show how degree and clustering develop for a single graph run. 
Interestingly, the results are unpredictable and the graphs look different each time. 
So we should experiment with sampling techniques and see if more regular distributions and patterns are observed. 
If the current system is chaotic or 'complex' it will be interesting to see what threshold reduces behaviour to a more regular pattern. 

Running this a few times shows chaotic results. Sometimes the graph gets completely disconnected and can't recover. This is because if there are no links at all, there is no rule to add a spontaneous belief. We could add one.

In [ ]:
// The 'iterate' function returns a sequence of network states 
// so we can iterate over that to calculate whetever stats
// we are interested in. Note that this is lazy sequence.
// Using the 'update' function from the previous
Sim.iterate 100 update g 
|> Seq.map clusteringCoeffcient
|> Seq.indexed 
|> Chart.Line
|> Chart.withTitle("Clustering coeffcient over 10000 iterations")



<!-- Plotly chart will be drawn inside this DIV --> 


 
</div

The average degree also currently looks chaotic. In about 1 out of 3 runs, edges become 0 after a few thousand iterations. I wonder what the tipping points are.

In [ ]:
Sim.iterate 1000 update g
|> Seq.map averageDegree
|> Seq.indexed 
|> Chart.Line
|> Chart.withTitle("Average degree over 10000 iterations")

<!-- Plotly chart will be drawn inside this DIV --> 


 
</div

### First attempt at some sampling
To see if any patterns are present across multiple runs I have added some very basic sampling.
Performance is bad when calculating averages by iteration, but histograms of final degree and clustering coeffcient are interesting.
They seem to increase when iterations are < 1000. However, the single run charts above seem to show this increase in density reverses at a later iteration. 
All very basic at this point but is a start towards using stats to guide the dev and interrupt the results.

Charts take over a minute to run on my laptop.



In [ ]:
/// Takes a number of samples and returns a seq of averages at each index(iteration). 
/// (I was hoping this would create more regular results but doesnt seem to at low sample volumes)
let averageFromSamples (samples: float[][])  = 
  Array.init (samples.[0].Length) (fun i -> 
    seq { for s in samples -> s.[i] } |> Seq.average)

// There are also gaps which needs investigation.
Array.init 500 (fun x -> 
  Sim.iterate 100 update g 
  |> Seq.map clusteringCoeffcient
  |> Array.ofSeq )
|> averageFromSamples
|> Seq.indexed 
|> Chart.Line
|> Chart.withTitle("Clustering coeffcient by iteration averaged over 500 samples")

<!-- Plotly chart will be drawn inside this DIV --> 


 
</div

In [ ]:
/// Returns the final state from all samples. 
let resultsFromSamples (samples:float[][])  =  
  [| for s in samples -> s.[s.Length - 1] |]
  
// Aven with Array.ofSeq, this one is pretty slow (~2 minutes)
Array.init 100 (fun x -> Sim.iterate 1000 update g |> Seq.map clusteringCoeffcient |> Array.ofSeq)
|> resultsFromSamples
|> Chart.Histogram
|> Chart.withTitle("Final clustering coeffcient after 1000 iterations over 100 samples")

<!-- Plotly chart will be drawn inside this DIV --> 


 
</div

In [ ]:
Array.init 100 (fun x -> Sim.iterate 100 update g |> Seq.map averageDegree |> Array.ofSeq)
|> averageFromSamples
|> Seq.indexed 
|> Chart.Line
|> Chart.withTitle("Average degree by iteration averaged over 100 samples")

<!-- Plotly chart will be drawn inside this DIV --> 


 
</div

In [ ]:
// seems very unpreditable
Array.init 100 (fun x -> Sim.iterate 1000 update g |> Seq.map averageDegree |> Array.ofSeq)
|> resultsFromSamples
|> Chart.Histogram
|> Chart.withTitle("Average degree after 1000 iterations over 100 samples ")

<!-- Plotly chart will be drawn inside this DIV --> 


 
</div